## 1. Import Libraries & Setup

In [ ]:
%pip install -q plotly

import sys
from pathlib import Path

# Add backend to path
sys.path.insert(0, str(Path.cwd() / 'backend'))

import pandas as pd
import numpy as np

try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
except ImportError:
    print("Installing plotly...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots

from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

## 2. Load Data

In [ ]:
try:
    from data_fetch import DataFetcher, load_demo_data
except ImportError as e:
    print(f"Error importing data_fetch: {e}")
    print("Make sure backend/data_fetch.py exists and sys.path includes backend directory")
    raise

# Load 5 years of historical data
print("Loading historical data for XAU/USD (Gold)...")
df = load_demo_data(years=5)

# Use recent data for quick analysis
df_analysis = df.tail(500).copy()

print(f"\nData Loaded:")
print(f"  Total Records: {len(df)}")
print(f"  Analysis Period: {df_analysis.index[0]} to {df_analysis.index[-1]}")
print(f"  Latest Price: ${df_analysis['Close'].iloc[-1]:.2f}")
print(f"\nData Preview:")
df_analysis.head(10)

## 3. Detect Zones, Order Blocks, FVGs

In [ ]:
try:
    from zone_detector import ZoneDetector
except ImportError as e:
    print(f"Error importing zone_detector: {e}")
    raise

print("Detecting zones and ICT elements...\n")

detector = ZoneDetector(lookback=20, atr_multiplier=1.5)

# Detect swings
swing_highs, swing_lows = detector.detect_swings(df_analysis)
print(f"Swing Highs: {len(swing_highs)}")
print(f"Swing Lows: {len(swing_lows)}")

# Detect zones
zones = detector.detect_supply_demand_zones(df_analysis, swing_highs, swing_lows)
zones = detector.update_zone_mitigation(zones, df_analysis)
print(f"\nZones Detected: {len(zones)}")

unmitigated = sum(1 for z in zones if not z.mitigated)
print(f"  Unmitigated: {unmitigated}")
print(f"  Mitigated: {len(zones) - unmitigated}")

# Detect Order Blocks
obs = detector.detect_order_blocks(df_analysis)
print(f"\nOrder Blocks: {len(obs)}")
bullish_obs = sum(1 for ob in obs if ob.ob_type == 'bullish')
bearish_obs = sum(1 for ob in obs if ob.ob_type == 'bearish')
print(f"  Bullish OBs: {bullish_obs}")
print(f"  Bearish OBs: {bearish_obs}")

# Detect Fair Value Gaps
fvgs = detector.detect_fair_value_gaps(df_analysis)
print(f"\nFair Value Gaps: {len(fvgs)}")
bullish_fvg = sum(1 for fvg in fvgs if fvg.fvg_type == 'bullish')
bearish_fvg = sum(1 for fvg in fvgs if fvg.fvg_type == 'bearish')
print(f"  Bullish FVGs: {bullish_fvg}")
print(f"  Bearish FVGs: {bearish_fvg}")

# Support/Resistance
srl = detector.detect_support_resistance(df_analysis, n_recent=5)
print(f"\nSupport Levels: {[f'{s:.2f}' for s in srl['support'][:3]]}")
print(f"Resistance Levels: {[f'{r:.2f}' for r in srl['resistance'][:3]]}")

## 4. Generate Trading Signals

In [ ]:
try:
    from signal_generator import SignalGenerator, SignalType
except ImportError as e:
    print(f"Error importing signal_generator: {e}")
    raise

print("Generating trading signals...\n")

generator = SignalGenerator(
    min_risk_reward=1.5,
    ema_fast=30,
    ema_slow=200
)

df_signals = generator.generate_signals(
    df_analysis,
    zones,
    obs,
    fvgs,
    srl,
    lookback=len(df_analysis)
)

# Count signals
buy_signals = len(df_signals[df_signals['Signal'] == SignalType.BUY.value])
sell_signals = len(df_signals[df_signals['Signal'] == SignalType.SELL.value])

print(f"Signals Generated:")
print(f"  Buy Signals: {buy_signals}")
print(f"  Sell Signals: {sell_signals}")
print(f"  Total Signals: {buy_signals + sell_signals}")

# Show recent signals
print("\nRecent Signals:")
signals_df = df_signals[df_signals['Signal'] != 0][['Close', 'Signal', 'EntryPrice', 'StopLoss', 'TakeProfit', 'RiskReward', 'Reason']].tail(10)
signals_df['Signal'] = signals_df['Signal'].apply(lambda x: 'BUY' if x == 1 else 'SELL')
signals_df

## 5. Run Backtest

In [ ]:
try:
    from backtester import SimpleBacktester
except ImportError as e:
    print(f"Error importing backtester: {e}")
    raise

print("Running backtest with realistic slippage & spread...\n")

backtester = SimpleBacktester(
    initial_capital=10000,
    position_size=0.95,
    slippage_pips=1.0,
    spread_pips=0.5
)

backtest_results = backtester.run_backtest(df_signals)

# Print report
report = backtester.print_backtest_report(backtest_results)
print(report)

## 6. Results Analysis

In [ ]:
# Extract trades for analysis
trades_list = backtest_results.get('trades', [])

if trades_list:
    trades_df = pd.DataFrame(trades_list)
    
    print("Trade Analysis:")
    print(f"\nFirst 5 Trades:")
    print(trades_df[['direction', 'entry_price', 'exit_price', 'pnl', 'return_pct']].head())
    
    print(f"\nLast 5 Trades:")
    print(trades_df[['direction', 'entry_price', 'exit_price', 'pnl', 'return_pct']].tail())
    
    print(f"\nTrade Statistics:")
    print(f"  Avg Entry-to-Exit Time: {trades_df['return_pct'].mean():.2f}%")
    print(f"  Median Return: {trades_df['return_pct'].median():.2f}%")
else:
    print("No completed trades in backtest")

## 7. Visualize Results - Equity Curve

In [ ]:
# Create equity curve from trades
if trades_list:
    trades_df = pd.DataFrame(trades_list)
    
    fig = go.Figure()
    
    # Add equity curve
    fig.add_trace(go.Scatter(
        x=range(len(trades_df)),
        y=trades_df['balance'],
        mode='lines+markers',
        name='Account Balance',
        line=dict(color='blue', width=2),
        marker=dict(size=4)
    ))
    
    fig.update_layout(
        title='Equity Curve - Backtest Results',
        xaxis_title='Trade Number',
        yaxis_title='Account Balance ($)',
        hovermode='x unified',
        template='plotly_white',
        height=500
    )
    
    fig.show()
else:
    print("No trades to visualize")

## 8. Visualize Price Action with Zones

In [ ]:
# Plot candlesticks with zones
plot_df = df_analysis.tail(100).reset_index()

fig = go.Figure(data=[
    go.Candlestick(
        x=plot_df['Datetime'],
        open=plot_df['Open'],
        high=plot_df['High'],
        low=plot_df['Low'],
        close=plot_df['Close'],
        name='XAU/USD'
    )
])

# Add EMAs
fig.add_trace(go.Scatter(
    x=plot_df['Datetime'],
    y=plot_df['EMA_30'],
    name='EMA 30',
    line=dict(color='blue', width=1)
))

fig.add_trace(go.Scatter(
    x=plot_df['Datetime'],
    y=plot_df['EMA_200'],
    name='EMA 200',
    line=dict(color='orange', width=1)
))

fig.update_layout(
    title='XAU/USD Price Action (Last 100 Candles)',
    xaxis_title='Date',
    yaxis_title='Price ($)',
    hovermode='x unified',
    template='plotly_white',
    height=600
)

fig.show()

## 9. Performance Metrics Summary

In [ ]:
# Create performance summary
metrics = {
    'Metric': ['Total Trades', 'Win Rate (%)', 'Profit Factor', 'Total P&L ($)', 'Max Drawdown (%)', 'Sharpe Ratio'],
    'Value': [
        backtest_results['total_trades'],
        f"{backtest_results['win_rate']:.2f}",
        f"{backtest_results['profit_factor']:.2f}",
        f"${backtest_results['total_pnl']:.2f}",
        f"{backtest_results['max_drawdown_pct']:.2f}",
        f"{backtest_results['sharpe_ratio']:.2f}"
    ],
    'Target': [
        '10+',
        '>55%',
        '>1.5',
        'Positive',
        '<20%',
        '>1.0'
    ]
}

metrics_df = pd.DataFrame(metrics)
print("\n" + "="*60)
print("BACKTEST PERFORMANCE METRICS")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)

## 10. Next Steps

✅ **Completed:**
- Data loading and validation
- Zone detection (supply/demand, OBs, FVGs)
- Signal generation with confluence
- Backtest with realistic costs
- Performance analysis

📋 **Recommended Next Steps:**

1. **Optimize Parameters**: Adjust `swing_lookback`, `atr_multiplier`, `min_risk_reward`
2. **Walk-Forward Analysis**: Test on 2024 data to validate robustness
3. **Deploy Paper Trading**: Use `forward_tester.py` on live data
4. **Setup Webhook**: Configure TradingView alerts to webhook server
5. **Monitor Live Performance**: Compare backtest vs. live trading
6. **Auto-Execution**: Integrate with broker API (OANDA, Interactive Brokers)

💡 **Key Insights:**
- If Win Rate < 55%: Check zone detection sensitivity (lower `swing_lookback`)
- If Profit Factor < 1.5: Increase `min_risk_reward` or improve SL/TP calculation
- If Max Drawdown > 20%: Reduce position size or add filters

⚠️ **Disclaimer:** Historical backtest results do not guarantee live performance. Trade at your own risk.